In [ ]:
# lstm,rnn, bilstm, bi-gru

In [ ]:
import kagglehub
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense, Dropout, Bidirectional, GRU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from tensorflow.keras.utils import to_categorical
import pickle

In [ ]:
dataset_path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")

train_file = f"{dataset_path}/train.csv"
test_file = f"{dataset_path}/test.csv"

100%|██████████| 11.4M/11.4M [00:00<00:00, 90.3MB/s]

Extracting files...


In [ ]:
df_train_full = pd.read_csv(train_file)
df_test_full = pd.read_csv(test_file)

In [ ]:
df_train_full.head(2)

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...


In [ ]:
df_test_full.head(2)

,Class Index,Title,Description
0,3,Fears for T N pension after talks,Unions representing workers at Turner Newall...
1,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o..."


In [ ]:
df_train_full['text'] = df_train_full['Title'].fillna('') + ' ' + df_train_full['Description'].fillna('')
df_test_full['text'] = df_test_full['Title'].fillna('') + ' ' + df_test_full['Description'].fillna('')

In [ ]:
df_train_full.head(2)

,Class Index,Title,Description,text
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...


In [ ]:
df_test_full.head(2)

,Class Index,Title,Description,text
0,3,Fears for T N pension after talks,Unions representing workers at Turner Newall...,Fears for T N pension after talks Unions repre...
1,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o...",The Race is On: Second Private Team Sets Launc...


In [ ]:
df_train_full['Class Index'].value_counts() #0,1,2,3

,count
Class Index,
3,30000
4,30000
2,30000
1,30000


In [ ]:
df_train_full['label'] = df_train_full['Class Index'].astype(int)-1
df_test_full['label'] = df_test_full['Class Index'].astype(int)-1

In [ ]:
df_train_full.head(3)

,Class Index,Title,Description,text,label
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...,2
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...,2


In [ ]:
df_train_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Class Index  120000 non-null  int64 
 1   Title        120000 non-null  object
 2   Description  120000 non-null  object
 3   text         120000 non-null  object
 4   label        120000 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 4.6+ MB


In [ ]:
train_samples = 3000
test_samples = 1000
df_train = df_train_full.iloc[:train_samples]
df_test = df_test_full.iloc[:test_samples]

In [ ]:
#see some smaple data
class_name = {0:"World",1:"Sports",2:"Business",3:"Sci/Tech"}
for i in range(3):
  print(f"Text : {df_train.iloc[i]['text']}")
  print(f"Label : {class_name[df_train.iloc[i]['label']]}\n")

Text : Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
Label : Business

Text : Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market.
Label : Business

Text : Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the outlook for earnings are expected to\hang over the stock market next week during the depth of the\summer doldrums.
Label : Business



In [ ]:
max_features = 5000 #vocab
maxlen = 500  #max seq length

In [ ]:
tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(df_train['text'])

In [ ]:
x_train_seq = tokenizer.texts_to_sequences(df_train['text'])
x_test_seq = tokenizer.texts_to_sequences(df_test['text'])

In [ ]:
x_train = pad_sequences(x_train_seq, maxlen=maxlen)
x_test = pad_sequences(x_test_seq,maxlen=maxlen)

In [ ]:
# 0,1,2,3 -> more 3 ,less 0

In [ ]:
y_train = to_categorical(df_train['label'],num_classes=4)
y_test_cat = to_categorical(df_test['label'],num_classes=4)
y_test = df_test['label'] #f1 score

In [ ]:
def train_and_evaluate_model(model, model_name, x_train, y_train, x_test, y_test_cat, y_test):
  print(f"\n{'='*50}")
  print(f"Training {model_name}")
  print(f"{'='*50}")

  model.compile(
      optimizer='adam',
      loss='categorical_crossentropy',
      metrics=['accuracy']
  )

  print(f"Starting Model Training.......\n")
  history = model.fit(
      x_train, y_train,
      epochs=5,
      batch_size=64,
      validation_split = 0.2,
      callbacks=[EarlyStopping(monitor='val_loss',patience=2, restore_best_weights=True,)]
  )

  print(f"Evaluating {model_name}\n")
  test_loss, test_acc = model.evaluate(x_test, y_test_cat)

  #f1 score
  y_pred_probs = model.predict(x_test)
  y_pred = np.argmax(y_pred_probs, axis=1)

  macro_f1 = f1_score(y_test, y_pred, average='macro')

  print(f"\n{model_name} Results:")
  print(f"Test Accuracy : {test_acc:.2f}")
  print(f"Macro F1-score : {macro_f1:.2f}")


  print(f"\nDetailed classification report for {model_name}:")
  target_names = ['World','Sports','Business','Sci/Tech']
  print(classification_report(y_test,y_pred,target_names=target_names))

  return {
      "model":model,
      "model_name":model_name,
      "history":history,
      "test_accuracy":test_acc,
      "macro_f1":macro_f1,
      "predictions":y_pred
  }

In [ ]:
all_results = []

In [ ]:
#model lstm -> stacking

In [ ]:
model_lstm = Sequential([
    Embedding(max_features, 64, input_length=maxlen),
    LSTM(64, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(4, activation='softmax')
    ])

result_lstm  = train_and_evaluate_model(model_lstm, "Simple Stacked LSTM",x_train,y_train,
                                        x_test, y_test_cat, y_test)
all_results.append(result_lstm)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



Training Simple Stacked LSTM
Starting Model Training.......

Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 13s 53ms/step - accuracy: 0.3825 - loss: 1.2937 - val_accuracy: 0.4000 - val_loss: 1.1915
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.7596 - loss: 0.6459 - val_accuracy: 0.8050 - val_loss: 0.5668
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.9038 - loss: 0.2986 - val_accuracy: 0.8417 - val_loss: 0.4737
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.9600 - loss: 0.1687 - val_accuracy: 0.8150 - val_loss: 0.5590
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step - accuracy: 0.9767 - loss: 0.1032 - val_accuracy: 0.7900 - val_loss: 0.6974
Evaluating Simple Stacked LSTM

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7950 - loss: 0.5763
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step

Simple Stacked LSTM Results:
Test Accuracy : 0.80
Macro F1-score : 0.79

Detailed classification report for Simple Stacked LSTM:
              precision    reca

In [ ]:
#bi-lstm - stacking - large data - 5000? 4000?
model_bilstm = Sequential([
    Embedding(max_features, 64, input_length=maxlen),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(4, activation='softmax')
    ])

result_bilstm  = train_and_evaluate_model(model_bilstm, "BI-LSTM",x_train,y_train,
                                        x_test, y_test_cat, y_test)
all_results.append(result_bilstm)


Training BI-LSTM
Starting Model Training.......

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


38/38 ━━━━━━━━━━━━━━━━━━━━ 8s 114ms/step - accuracy: 0.3917 - loss: 1.3052 - val_accuracy: 0.3900 - val_loss: 1.2030
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step - accuracy: 0.6938 - loss: 0.7567 - val_accuracy: 0.7667 - val_loss: 0.6557
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 75ms/step - accuracy: 0.9025 - loss: 0.3376 - val_accuracy: 0.8083 - val_loss: 0.6262
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 74ms/step - accuracy: 0.9492 - loss: 0.1811 - val_accuracy: 0.8300 - val_loss: 0.5729
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 89ms/step - accuracy: 0.9625 - loss: 0.1376 - val_accuracy: 0.7650 - val_loss: 0.8834
Evaluating BI-LSTM

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.7730 - loss: 0.6947
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 38ms/step

BI-LSTM Results:
Test Accuracy : 0.77
Macro F1-score : 0.77

Detailed classification report for BI-LSTM:
              precision    recall  f1-score   support

       World       0.80      0.76      0.78       268
      Sports       0.87      0

In [ ]:
#gru
model_gru = Sequential([
    Embedding(max_features, 64, input_length=maxlen),
    GRU(64, return_sequences=True),
    Dropout(0.3),
    GRU(64),
    Dropout(0.3),
    Dense(4, activation='softmax')
    ])

result_gru  = train_and_evaluate_model(model_gru, "Stacked GRU",x_train,y_train,
                                        x_test, y_test_cat, y_test)
all_results.append(result_gru)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



Training Stacked GRU
Starting Model Training.......

Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 10s 98ms/step - accuracy: 0.3496 - loss: 1.3448 - val_accuracy: 0.2117 - val_loss: 1.4583
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 158ms/step - accuracy: 0.6187 - loss: 0.9130 - val_accuracy: 0.6433 - val_loss: 0.8427
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 120ms/step - accuracy: 0.8625 - loss: 0.3697 - val_accuracy: 0.7783 - val_loss: 0.6779
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 85ms/step - accuracy: 0.9508 - loss: 0.1438 - val_accuracy: 0.7300 - val_loss: 0.9149
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 75ms/step - accuracy: 0.9808 - loss: 0.0769 - val_accuracy: 0.7850 - val_loss: 0.7466
Evaluating Stacked GRU

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - accuracy: 0.7300 - loss: 0.7436
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step

Stacked GRU Results:
Test Accuracy : 0.73
Macro F1-score : 0.73

Detailed classification report for Stacked GRU:
              precision    recall  f1-score   support

      

In [ ]:
model_bigru = Sequential([
    Embedding(max_features, 64, input_length=maxlen),
    Bidirectional(GRU(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(GRU(64)),
    Dropout(0.3),
    Dense(4, activation='softmax')
    ])

result_bigru  = train_and_evaluate_model(model_bigru, "BI-GRU",x_train,y_train,
                                        x_test, y_test_cat, y_test)
all_results.append(result_bigru)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



Training BI-GRU
Starting Model Training.......

Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 12s 139ms/step - accuracy: 0.3533 - loss: 1.3558 - val_accuracy: 0.2117 - val_loss: 1.3996
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 8s 75ms/step - accuracy: 0.5604 - loss: 0.9835 - val_accuracy: 0.5817 - val_loss: 0.8785
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 73ms/step - accuracy: 0.8625 - loss: 0.3895 - val_accuracy: 0.7233 - val_loss: 0.7540
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 73ms/step - accuracy: 0.9517 - loss: 0.1547 - val_accuracy: 0.7333 - val_loss: 0.8056
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 141ms/step - accuracy: 0.9787 - loss: 0.0775 - val_accuracy: 0.7467 - val_loss: 0.9105
Evaluating BI-GRU

32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.7340 - loss: 0.7813
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step

BI-GRU Results:
Test Accuracy : 0.73
Macro F1-score : 0.72

Detailed classification report for BI-GRU:
              precision    recall  f1-score   support

       World       0.68   

Q1. The Universal Approximation Theorem states that a feedforward network with a single hidden layer can approximate any continuous function on a compact domain. What is the key practical limitation this theorem does not address?


- It only applies to classification, not regression
- It guarantees existence but says nothing about the number of neurons required or whether training will find the weights
- It requires the activation function to be linear
- It only holds for networks with at least three hidden layers

In [ ]:
# It guarantees existence but says nothing about the number of neurons required or whether training will find the weights

Q2. Why does stacking multiple linear layers without nonlinear activations provide no more representational power than a single linear layer?


- A) Because gradients vanish through linear layers
- B) Because the composition of linear transformations is itself a single linear transformation
- C) Because biases cancel out across layers
- D) Because linear layers cannot be initialized randomly

In [ ]:
# Anwer - B

A dense (fully connected) layer maps a 256-dimensional input to a 128-dimensional output. How many trainable parameters does it have (including biases)?


- A) 32,768
- B) 32,896
- C) 384
- D) 33,024

In [ ]:
# Answer = 256 * 128 = 32,768 + 128 = 32,896

Q4. When the sigmoid activation is used throughout a deep MLP, why are gradients prone to vanishing as depth increases?


- A) The sigmoid output is always negative
- B) The maximum value of the sigmoid's derivative is 0.25, so repeated multiplication during backprop shrinks gradients exponentially
- C) Sigmoid has no derivative at 0
- D) Sigmoid forces all weights to zero

In [ ]:
# B ->

Q5. Why is initializing all weights of a hidden layer to the same value (e.g., zero) a poor choice?


- A) It causes immediate overfitting
- B) All neurons in a layer compute identical outputs and receive identical gradients, so they never differentiate ("symmetry is never broken")
- C) It makes the loss function non-differentiable
- D) It always produces exploding gradients

In [ ]:
# B

Q6. In backpropagation, what is actually being computed and propagated backward through the network?


- A) The raw activations of each layer
- B) The gradient of the loss with respect to each parameter, via the chain rule
- C) The updated weights themselves
- D) The second derivative (Hessian) of the loss

In [ ]:
#B

Q7. Compared to a fully connected layer on the same image, a convolutional layer dramatically reduces parameters primarily because of:


- A) Pooling layers
- B) Parameter sharing — the same kernel weights slide across all spatial positions
- C) Using ReLU instead of sigmoid
- D) Smaller batch sizes

Q8 Like the Bi-LSTM, a Bi-GRU is unsuitable for which use case, and for the same reason?


- Short sequences — because it underfits
- Offline document classification — because it overfits
- Real-time / streaming or autoregressive generation — because the backward pass needs the entire sequence before producing outputs
- Sequence labeling — because it ignores context

Which scenario makes a Bi-LSTM inappropriate?

- Named-entity recognition on complete sentences
- Offline sentiment classification
- Real-time / streaming prediction or autoregressive generation where future tokens aren't available yet
- Part-of-speech tagging on full documents

In [ ]:
!pip install presidio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 3.4 MB/s eta 0:00:00


In [ ]:
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import RecognizerResult, OperatorConfig

# Initialize the engine:
engine = AnonymizerEngine()

# Invoke the anonymize function with the text,
# analyzer results (potentially coming from presidio-analyzer) and
# Operators to get the anonymization output:
result = engine.anonymize(
    text="My name is Prasad and my friens is Jhansi",
    analyzer_results=[
        RecognizerResult(entity_type="PERSON", start=11, end=15, score=0.8),
        RecognizerResult(entity_type="PERSON", start=17, end=27, score=0.8),
    ],
    operators={"PERSON": OperatorConfig("replace", {"new_value": "[MASK]"})},
)

print(result)

text: My name is [MASK]ad[MASK]iens is Jhansi
items:
[
    {'start': 19, 'end': 25, 'entity_type': 'PERSON', 'text': '[MASK]', 'operator': 'replace'},
    {'start': 11, 'end': 17, 'entity_type': 'PERSON', 'text': '[MASK]', 'operator': 'replace'}
]

